# Multi-tenant CT capture read + bronze MERGE (workstream A) — IMPLEMENTED

Runs on the DBSQL **warehouse** (SQL only). Reads `CHANGETABLE(CHANGES <schema>.<table>, :ct_version)`
`LEFT JOIN` the base table on the PK via `remote_query` (PK projected from `ct` so deletes survive
as NULL base columns), exposes it as `vw_src`, and MERGEs into the bronze sink keyed on the PK
(idempotent net-change: `D`->DELETE, matched->UPDATE non-PK, not-matched->INSERT).

The inner query is passed to `remote_query` as a bound parameter (`:q`) — no quote-escaping — and
carries a per-run `uuid()` comment: unique across cycles (fresh read, defeats remote_query's stale
cache) yet identical within this run, so repeated MERGE scans see one consistent snapshot.

Builder UDFs below mirror the `get_join_clause` idiom in
`src/lakefed_ingest/copy_data_incremental.ipynb`; they are unit-tested by
`multitenant/tests/ct_builders_test.py` (extracted from this notebook).

In [ ]:
-- Query-string builders (idiom: copy_data_incremental.ipynb get_join_clause).
-- Unit-tested by multitenant/tests/ct_builders_test.py (bodies extracted from here).
CREATE OR REPLACE TEMPORARY FUNCTION ct_projection(select_list STRING, primary_key STRING)
RETURNS STRING LANGUAGE PYTHON AS $$
pk = {c.strip() for c in primary_key.split(',')}
cols = [c.strip() for c in select_list.split(',')]
parts = [(f'ct.[{c}] AS [{c}]' if c in pk else f't.[{c}] AS [{c}]') for c in cols]
parts.append('ct.SYS_CHANGE_OPERATION AS sys_change_operation')
return ', '.join(parts)
$$;

CREATE OR REPLACE TEMPORARY FUNCTION ct_join(primary_key STRING)
RETURNS STRING LANGUAGE PYTHON AS $$
return ' AND '.join(f't.[{c.strip()}] = ct.[{c.strip()}]' for c in primary_key.split(','))
$$;

CREATE OR REPLACE TEMPORARY FUNCTION merge_on(primary_key STRING)
RETURNS STRING LANGUAGE PYTHON AS $$
return ' AND '.join(f'tgt.`{c.strip()}` = s.`{c.strip()}`' for c in primary_key.split(','))
$$;

CREATE OR REPLACE TEMPORARY FUNCTION merge_set(select_list STRING, primary_key STRING)
RETURNS STRING LANGUAGE PYTHON AS $$
pk = {c.strip() for c in primary_key.split(',')}
cols = [c.strip() for c in select_list.split(',')]
non = [c for c in cols if c not in pk] or cols
return ', '.join(f'tgt.`{c}` = s.`{c}`' for c in non)
$$;

CREATE OR REPLACE TEMPORARY FUNCTION merge_insert_cols(select_list STRING)
RETURNS STRING LANGUAGE PYTHON AS $$
return ', '.join(f'`{c.strip()}`' for c in select_list.split(','))
$$;

CREATE OR REPLACE TEMPORARY FUNCTION merge_insert_vals(select_list STRING)
RETURNS STRING LANGUAGE PYTHON AS $$
return ', '.join(f's.`{c.strip()}`' for c in select_list.split(','))
$$;

In [ ]:
-- Build the CT read (T-SQL) and expose it as vw_src via remote_query (bound :q, uuid-hardened).
DECLARE OR REPLACE inner_q STRING;
SET VAR inner_q =
    'SELECT ' || ct_projection(:select_list, :primary_key)
    || ' FROM CHANGETABLE(CHANGES ' || :src_schema || '.' || :src_table || ', ' || :ct_version || ') AS ct'
    || ' LEFT JOIN ' || :src_schema || '.' || :src_table || ' AS t ON ' || ct_join(:primary_key)
    || ' /* ' || uuid() || ' */';

EXECUTE IMMEDIATE
    'CREATE OR REPLACE TEMPORARY VIEW vw_src AS SELECT * FROM remote_query(:c, database => :d, query => :q)'
    USING :src_connection AS c, :src_database AS d, inner_q AS q;

In [ ]:
-- Idempotent net-change MERGE into the bronze sink (I/U/D on sys_change_operation).
DECLARE OR REPLACE merge_q STRING;
SET VAR merge_q =
    'MERGE INTO ' || :sink_catalog || '.' || :sink_schema || '.' || :sink_table || ' AS tgt '
    || 'USING vw_src AS s ON ' || merge_on(:primary_key) || ' '
    || 'WHEN MATCHED AND s.sys_change_operation = ' || chr(39) || 'D' || chr(39) || ' THEN DELETE '
    || 'WHEN MATCHED THEN UPDATE SET ' || merge_set(:select_list, :primary_key) || ' '
    || 'WHEN NOT MATCHED AND s.sys_change_operation <> ' || chr(39) || 'D' || chr(39) || ' THEN INSERT ('
    || merge_insert_cols(:select_list) || ') VALUES (' || merge_insert_vals(:select_list) || ')';

EXECUTE IMMEDIATE merge_q;